# TriageAgent — GRPO Training

> **Hackathon:** Meta PyTorch × OpenEnv | **Theme:** World Modeling + Long-Horizon Planning

## What this notebook does

Trains **Qwen2.5-3B-Instruct** with **GRPO** (Group Relative Policy Optimization) to resolve
enterprise IT support tickets via multi-turn tool use.

### The task
The agent receives a new IT ticket and must investigate it by calling 7 MCP-style tools
that mirror real Atlassian/Confluence APIs:

| Tool | What it does |
|---|---|
| `search_kb` | Full-text search over 150 KB runbooks |
| `get_article` | Fetch a full KB article by ID |
| `search_tickets` | Search 60 past resolved tickets |
| `get_ticket` | Fetch a full past ticket with comments |
| `search_incidents` | Search 25 incident postmortems |
| `get_incident` | Fetch a full postmortem |
| `submit_resolution` | Submit the final answer (ends the episode) |

The agent gets 20 tool calls per ticket. It must decide what to search, what to read,
and when it has enough evidence — a capability current LLMs struggle with.

### Reward
| Component | Weight | What it measures |
|---|---|---|
| Primary (binary) | 1.0 | Resolution correct **and** citations grounded |
| Grounding | 0.3 | Citation F1 (partial credit) |
| Efficiency | 0.2 | Not over-searching relative to difficulty |
| Calibration | 0.15 | Confidence matches correctness (Brier-style) |

Expected trajectory: primary 0.25 → 0.60 over 200 GRPO steps.

## GPU requirements

| GPU | Status |
|---|---|
| **A100 40 GB** | ✅ Preferred. Full run (200 steps) ~90 min |
| A100 80 GB | ✅ Works. Can increase `num_generations` to 8 |
| L4 24 GB | ⚠️ Marginal. Reduce `num_generations=2`, `max_completion_length=2048` |
| T4 16 GB | ❌ Will OOM. vLLM + 3B model + GRPO does not fit |

**In Colab:** Runtime → Change runtime type → A100.
You need Colab Pro or compute credits for A100 access.

## SMOKE_TEST flag

`SMOKE_TEST = True` (default) runs **5 training steps** to verify the entire pipeline
wires together correctly without spending GPU time or API credits.

**For judges evaluating the full training run:**
Set `SMOKE_TEST = False` in the cell below before clicking *Run All*.
Full run = 200 steps ≈ 90 minutes on an A100.

In [1]:
# ── Configuration ────────────────────────────────────────────────────────────
SMOKE_TEST = True          # Set False for the full 200-step run

REPO_URL = "https://github.com/yahid/triage_agent_env"  # <-- update this
MODEL    = "Qwen/Qwen2.5-3B-Instruct"

# trackio API key (free at trackio.io — leave empty to skip logging)
TRACKIO_API_KEY = ""       # or set via: import os; os.environ['TRACKIO_API_KEY'] = '...'

# ── Checkpoint resume ─────────────────────────────────────────────────────────
# Set to a checkpoint path to resume training from that step instead of zero.
# Example: RESUME_FROM_CHECKPOINT = "/content/triage_grpo_qwen3b/checkpoint-125"
# If training dies at step 150, checkpoints exist at 25, 50, 75, 100, 125.
# Set RESUME_FROM_CHECKPOINT = "/content/triage_grpo_qwen3b/checkpoint-125" to resume there.
RESUME_FROM_CHECKPOINT = None  # None = start from scratch

# ── HF Hub checkpoint backup ──────────────────────────────────────────────────
# Every 25 steps the checkpoint is pushed here automatically.
# A Colab crash loses at most 25 steps of compute, not the whole run.
# Set to None to disable hub push (e.g. if you haven't logged in to HF).
HF_HUB_REPO_ID = "yahid/triage-agent-qwen3b"  # <-- update to your HF username

MAX_STEPS = 5 if SMOKE_TEST else 200
print(f"SMOKE_TEST={SMOKE_TEST}  MAX_STEPS={MAX_STEPS}")
print(f"RESUME_FROM_CHECKPOINT={RESUME_FROM_CHECKPOINT}")
print(f"HF_HUB_REPO_ID={HF_HUB_REPO_ID}")

SMOKE_TEST=True  MAX_STEPS=5
RESUME_FROM_CHECKPOINT=None
HF_HUB_REPO_ID=yahid/triage-agent-qwen3b


## 1. Install dependencies

In [2]:
%%capture install_out
# Core training stack
!pip install -q \
    "trl>=0.12" \
    "transformers>=5.2" \
    "datasets>=2.19" \
    "accelerate>=0.30" \
    "bitsandbytes>=0.43" \
    "peft>=0.11" \
    "vllm>=0.4" \
    trackio \
    matplotlib

# Environment dependencies (lighter than full install)
!pip install -q numpy scikit-learn python-dotenv

print("Install complete.")

## 2. Clone repository

In [ ]:
import os, sys

REPO_DIR = "/content/triage_agent_env"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --ff-only

# Add project root to Python path
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

%cd {REPO_DIR}
print("Working directory:", os.getcwd())

## 3. Imports

In [ ]:
import json
from pathlib import Path
from typing import List, Optional

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
from datasets import Dataset
from trl import GRPOConfig, GRPOTrainer

from server.corpus import Corpus
from server.rewards import (
    primary_reward,
    reward_calibration,
    reward_efficiency,
    reward_grounding,
)

DATA_DIR   = Path(REPO_DIR) / "data"
ASSETS_DIR = Path(REPO_DIR) / "assets"
PLOTS_DIR  = ASSETS_DIR / "plots"
OUTPUT_DIR = Path("/content/triage_grpo_qwen3b")

PLOTS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Imports OK")

## 4. In-process training environment

TRL's `environment_factory` expects a class whose **public methods** become tools.
Docstrings with `Args:` blocks are parsed into OpenAI-format tool schemas automatically.
The trainer calls `reset(**dataset_row)` to start each episode and reads
the `reward` property after the episode ends.

In [ ]:
class _State:
    """Minimal mutable episode state — duck-typed to match rewards.py."""
    __slots__ = (
        "target_ticket_id", "gold_resolution", "gold_cited_ids",
        "difficulty", "is_unanswerable",
        "tools_called", "artifacts_viewed",
        "searches_made", "fetches_made",
        "submitted", "submitted_resolution", "submitted_citations",
        "submitted_confidence", "submitted_escalate",
    )
    def __init__(self):
        self.target_ticket_id = ""
        self.gold_resolution = ""
        self.gold_cited_ids = []
        self.difficulty = "medium"
        self.is_unanswerable = False
        self.tools_called = []
        self.artifacts_viewed = []
        self.searches_made = 0
        self.fetches_made = 0
        self.submitted = False
        self.submitted_resolution = None
        self.submitted_citations = []
        self.submitted_confidence = None
        self.submitted_escalate = False


class TriageEnvForTraining:
    """TRL environment_factory class for TriageAgent."""

    def __init__(self, corpus: Corpus, ticket_index: dict):
        self._corpus = corpus
        self._ticket_index = ticket_index
        self._state = _State()

    def reset(self, ticket_id: str = "", **kwargs) -> str:
        """Start a new triage episode."""
        ticket = self._ticket_index.get(ticket_id)
        if ticket is None:
            ticket = next(iter(self._ticket_index.values()))
        s = _State()
        s.target_ticket_id = ticket.get("ticket_id", ticket.get("id", ""))
        s.gold_resolution   = ticket.get("gold_resolution", "")
        s.gold_cited_ids    = list(ticket.get("gold_cited_ids", []))
        s.difficulty        = ticket.get("difficulty", "medium")
        s.is_unanswerable   = ticket.get("is_unanswerable", False)
        self._state = s
        return (
            f"# New Ticket: {s.target_ticket_id}\n\n"
            f"**Title:** {ticket.get('title', '')}\n\n"
            f"**Description:** {ticket.get('description', '')}\n\n"
            "You have 20 tool calls. Investigate, then call submit_resolution."
        )

    @property
    def reward(self) -> float:
        """Total episode reward — read by TRL after episode ends."""
        return (
            primary_reward(self._state)
            + reward_grounding(self._state)
            + reward_efficiency(self._state)
            + reward_calibration(self._state)
        )

    # ── Tools ────────────────────────────────────────────────────────────────

    def search_kb(self, query: str, max_results: int = 5) -> str:
        """Search the knowledge base for articles matching a query.

        Args:
            query: Natural language search query.
            max_results: Maximum number of results to return (1-10).
        """
        self._state.searches_made += 1
        self._state.tools_called.append(("search_kb", {"query": query}))
        return json.dumps(self._corpus.search_kb(query=query, max_results=max_results))

    def get_article(self, article_id: str) -> str:
        """Retrieve the full body of a KB article by ID.

        Args:
            article_id: The article identifier, e.g. 'KB-00042'.
        """
        self._state.fetches_made += 1
        self._state.artifacts_viewed.append(article_id)
        self._state.tools_called.append(("get_article", {"article_id": article_id}))
        r = self._corpus.get_article(article_id=article_id)
        return json.dumps(r) if r else '{"error": "Not found."}'

    def search_tickets(self, query: str, status: Optional[str] = None, max_results: int = 5) -> str:
        """Search past resolved tickets for similar issues.

        Args:
            query: Natural language search query.
            status: Filter by ticket status, e.g. 'Resolved'.
            max_results: Maximum number of results to return (1-10).
        """
        self._state.searches_made += 1
        self._state.tools_called.append(("search_tickets", {"query": query}))
        return json.dumps(self._corpus.search_tickets(query=query, status=status, max_results=max_results))

    def get_ticket(self, ticket_id: str) -> str:
        """Retrieve a full past ticket including comments and resolution.

        Args:
            ticket_id: The ticket identifier, e.g. 'TKT-000123'.
        """
        self._state.fetches_made += 1
        self._state.artifacts_viewed.append(ticket_id)
        self._state.tools_called.append(("get_ticket", {"ticket_id": ticket_id}))
        r = self._corpus.get_ticket(ticket_id=ticket_id)
        return json.dumps(r) if r else '{"error": "Not found."}'

    def search_incidents(self, query: str, max_results: int = 3) -> str:
        """Search incident postmortems for related outages or failures.

        Args:
            query: Natural language search query.
            max_results: Maximum number of results to return (1-5).
        """
        self._state.searches_made += 1
        self._state.tools_called.append(("search_incidents", {"query": query}))
        return json.dumps(self._corpus.search_incidents(query=query, max_results=max_results))

    def get_incident(self, incident_id: str) -> str:
        """Retrieve a full incident postmortem by ID.

        Args:
            incident_id: The incident identifier, e.g. 'INC-1234'.
        """
        self._state.fetches_made += 1
        self._state.artifacts_viewed.append(incident_id)
        self._state.tools_called.append(("get_incident", {"incident_id": incident_id}))
        r = self._corpus.get_incident(incident_id=incident_id)
        return json.dumps(r) if r else '{"error": "Not found."}'

    def submit_resolution(
        self,
        resolution: str,
        cited_artifacts: List[str],
        confidence: float,
        escalate: bool = False,
    ) -> str:
        """Submit your final resolution. This ends the episode.

        Args:
            resolution: The resolution text to send to the requester.
            cited_artifacts: List of artifact IDs supporting the resolution.
            confidence: Your confidence in the resolution, between 0.0 and 1.0.
            escalate: Set True if this ticket cannot be resolved with available info.
        """
        self._state.submitted             = True
        self._state.submitted_resolution  = resolution
        self._state.submitted_citations   = list(cited_artifacts)
        self._state.submitted_confidence  = float(confidence)
        self._state.submitted_escalate    = bool(escalate)
        self._state.tools_called.append(("submit_resolution", {}))
        return "Resolution submitted. Episode complete."


print("TriageEnvForTraining defined.")

## 5. Load data

In [ ]:
corpus = Corpus(DATA_DIR)

with open(DATA_DIR / "train_tickets.json") as f:
    train_tickets = json.load(f)

ticket_index = {t.get("ticket_id", t.get("id", "")): t for t in train_tickets}

# Build HuggingFace dataset — each row = one episode
dataset = Dataset.from_list([
    {
        "prompt": [
            {"role": "user", "content": "You are an enterprise IT triage agent. Resolve the ticket."}
        ],
        "ticket_id": t.get("ticket_id", t.get("id", "")),
    }
    for t in train_tickets
])

print(f"Corpus loaded. Train tickets: {len(train_tickets)}")
print(f"Dataset: {len(dataset)} rows")

# Smoke-test: verify env reset works
test_env = TriageEnvForTraining(corpus, ticket_index)
first_id = list(ticket_index.keys())[0]
obs = test_env.reset(ticket_id=first_id)
print("\nSample reset observation (first 300 chars):")
print(obs[:300])

## 6. Training

Key hyperparameter choices:
- **`num_generations=4`** — G=4 GRPO groups; memory-safe on A100 40 GB
- **`loss_type="dr_grpo"`** — fixes the length bias in vanilla GRPO
- **`vllm_mode="colocate"`** — generation and training share the single GPU
- **`vllm_gpu_memory_utilization=0.3`** — reserves 70% VRAM for training

In [ ]:
# Per-group reward functions — TRL passes a list of env instances after each rollout
def r_primary(environments, **kwargs):
    return [primary_reward(e._state) for e in environments]

def r_grounding(environments, **kwargs):
    return [reward_grounding(e._state) for e in environments]

def r_efficiency(environments, **kwargs):
    return [reward_efficiency(e._state) for e in environments]

def r_calibration(environments, **kwargs):
    return [reward_calibration(e._state) for e in environments]

# Environment factory — called once per parallel rollout slot
def env_factory():
    return TriageEnvForTraining(corpus, ticket_index)

print("Reward functions and env_factory defined.")

In [ ]:
# trackio login (optional — skip if you don't have an account)
if TRACKIO_API_KEY:
    import os
    os.environ["TRACKIO_API_KEY"] = TRACKIO_API_KEY

try:
    import trackio
    trackio.init(project="triage-agent-grpo")
    REPORT_TO = "trackio"
    print("trackio logging enabled.")
except Exception as e:
    REPORT_TO = "none"
    print(f"trackio not available ({e}). Logging disabled.")

# save_steps=25: checkpoint every 25 steps regardless of total run length.
# With max_steps=200 this gives 8 recovery points — losing a Colab session
# costs at most 25 steps (~11 min on A100) rather than the whole run.
SAVE_STEPS = 1 if SMOKE_TEST else 25

training_args = GRPOConfig(
    output_dir=str(OUTPUT_DIR),
    num_generations=4,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    max_prompt_length=2048,
    max_completion_length=4096,
    learning_rate=5e-6,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    optim="adamw_8bit",
    loss_type="dr_grpo",
    max_steps=MAX_STEPS,
    save_steps=SAVE_STEPS,
    logging_steps=1,
    bf16=True,
    use_vllm=True,
    vllm_mode="colocate",
    vllm_gpu_memory_utilization=0.3,
    report_to=REPORT_TO,
    # HF Hub backup — checkpoints are pushed at every save_steps
    push_to_hub=HF_HUB_REPO_ID is not None,
    hub_model_id=HF_HUB_REPO_ID,
    hub_strategy="checkpoint",        # push each checkpoint, not just the final model
    hub_private_repo=True,
)

trainer = GRPOTrainer(
    model=MODEL,
    reward_funcs=[r_primary, r_grounding, r_efficiency, r_calibration],
    reward_weights=[1.0, 0.3, 0.2, 0.15],
    environment_factory=env_factory,
    train_dataset=dataset,
    args=training_args,
)

print(f"Trainer ready. Training for {MAX_STEPS} steps, checkpoint every {SAVE_STEPS}.")
if HF_HUB_REPO_ID:
    print(f"Checkpoints will be pushed to: https://huggingface.co/{HF_HUB_REPO_ID}")
if RESUME_FROM_CHECKPOINT:
    print(f"Will resume from: {RESUME_FROM_CHECKPOINT}")

In [ ]:
trainer.train(resume_from_checkpoint=RESUME_FROM_CHECKPOINT)

## 7. Reward curves

Plot each sub-reward over training steps from the trainer's log history.

In [ ]:
log_history = trainer.state.log_history

# Extract per-step metrics
steps, r_prim, r_grnd, r_eff, r_cal, r_total = [], [], [], [], [], []

reward_keys = {
    "r_primary":    (r_prim,  "Primary (binary)"),
    "r_grounding":  (r_grnd,  "Grounding (×0.3)"),
    "r_efficiency": (r_eff,   "Efficiency (×0.2)"),
    "r_calibration":(r_cal,   "Calibration (×0.15)"),
}

for entry in log_history:
    if "loss" not in entry:
        continue  # skip eval-only entries
    steps.append(entry.get("step", 0))
    for key, (lst, _) in reward_keys.items():
        lst.append(entry.get(key, float("nan")))
    # total = weighted sum matching GRPOTrainer weights
    p = entry.get("r_primary", 0)
    g = entry.get("r_grounding", 0)
    e = entry.get("r_efficiency", 0)
    c = entry.get("r_calibration", 0)
    r_total.append(1.0*p + 0.3*g + 0.2*e + 0.15*c)

if not steps:
    print("No training logs found (SMOKE_TEST may have completed too fast). Skipping plots.")
else:
    steps = np.array(steps)

    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    fig.suptitle("TriageAgent GRPO — Sub-reward Curves", fontsize=14, fontweight="bold")

    for ax, (key, (vals, label)) in zip(axes.flat, reward_keys.items()):
        ax.plot(steps, vals, linewidth=1.5, alpha=0.7, color="steelblue")
        # 5-step rolling mean
        if len(vals) >= 5:
            rm = np.convolve(vals, np.ones(5)/5, mode="valid")
            ax.plot(steps[4:], rm, linewidth=2.0, color="crimson", label="5-step avg")
            ax.legend(fontsize=8)
        ax.set_title(label, fontsize=11)
        ax.set_xlabel("Training step")
        ax.set_ylabel("Reward")
        ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
        ax.grid(alpha=0.3)

    plt.tight_layout()
    plot_path = PLOTS_DIR / "training_curves.png"
    plt.savefig(plot_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {plot_path}")

    # Total reward summary
    fig2, ax2 = plt.subplots(figsize=(10, 4))
    ax2.plot(steps, r_total, linewidth=1.5, alpha=0.7, color="steelblue", label="Total reward")
    if len(r_total) >= 5:
        rm = np.convolve(r_total, np.ones(5)/5, mode="valid")
        ax2.plot(steps[4:], rm, linewidth=2.5, color="crimson", label="5-step avg")
    ax2.set_title("Total Episode Reward over Training", fontsize=13, fontweight="bold")
    ax2.set_xlabel("Training step")
    ax2.set_ylabel("Total reward")
    ax2.legend()
    ax2.grid(alpha=0.3)
    plt.tight_layout()
    total_plot_path = PLOTS_DIR / "training_total_reward.png"
    plt.savefig(total_plot_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {total_plot_path}")

## 8. Save LoRA adapter

In [ ]:
adapter_path = str(OUTPUT_DIR / "lora_adapter")
trainer.save_model(adapter_path)
print(f"LoRA adapter saved to: {adapter_path}")

# List saved files
import os
for f in sorted(os.listdir(adapter_path)):
    size_kb = os.path.getsize(os.path.join(adapter_path, f)) // 1024
    print(f"  {f:40s}  {size_kb:>8} KB")

## 9. Evaluation — trained model vs baseline

Run the **trained LoRA model** against `data/eval_tickets.json` (20 tickets) and
compare mean rewards to the pre-training baseline stored in `assets/baseline_eval.json`.

This uses local GPU inference — no API calls needed.

In [ ]:
import re
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# Load trained model
print("Loading trained model...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
trained_model = PeftModel.from_pretrained(base_model, adapter_path)
tokenizer = AutoTokenizer.from_pretrained(MODEL)
trained_model.eval()
print("Model loaded.")

# Load eval tickets
with open(DATA_DIR / "eval_tickets.json") as f:
    eval_tickets = json.load(f)
print(f"Eval tickets: {len(eval_tickets)}")

In [ ]:
# Tool definitions for chat template
TOOLS = [
    {"type": "function", "function": {"name": "search_kb",
     "parameters": {"type": "object", "properties": {
         "query": {"type": "string"}, "max_results": {"type": "integer"}},
     "required": ["query"]}}},
    {"type": "function", "function": {"name": "get_article",
     "parameters": {"type": "object", "properties": {
         "article_id": {"type": "string"}}, "required": ["article_id"]}}},
    {"type": "function", "function": {"name": "search_tickets",
     "parameters": {"type": "object", "properties": {
         "query": {"type": "string"}, "status": {"type": "string"},
         "max_results": {"type": "integer"}}, "required": ["query"]}}},
    {"type": "function", "function": {"name": "get_ticket",
     "parameters": {"type": "object", "properties": {
         "ticket_id": {"type": "string"}}, "required": ["ticket_id"]}}},
    {"type": "function", "function": {"name": "search_incidents",
     "parameters": {"type": "object", "properties": {
         "query": {"type": "string"}, "max_results": {"type": "integer"}},
     "required": ["query"]}}},
    {"type": "function", "function": {"name": "get_incident",
     "parameters": {"type": "object", "properties": {
         "incident_id": {"type": "string"}}, "required": ["incident_id"]}}},
    {"type": "function", "function": {"name": "submit_resolution",
     "parameters": {"type": "object", "properties": {
         "resolution": {"type": "string"},
         "cited_artifacts": {"type": "array", "items": {"type": "string"}},
         "confidence": {"type": "number"},
         "escalate": {"type": "boolean"}},
     "required": ["resolution", "cited_artifacts", "confidence"]}}},
]


def generate_reply(messages: list, max_new_tokens: int = 512) -> str:
    """Generate one reply from the trained model."""
    text = tokenizer.apply_chat_template(
        messages, tools=TOOLS, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(trained_model.device)
    with torch.no_grad():
        out = trained_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)


def parse_tool_call(text: str):
    """Extract tool name and JSON args from model output. Returns (name, args) or None."""
    match = re.search(r'<tool_call>\s*(\{.*?\})\s*</tool_call>', text, re.DOTALL)
    if not match:
        # fallback: look for JSON with 'name' key
        match = re.search(r'\{[^{}]*"name"\s*:\s*"([^"]+)"[^{}]*\}', text)
        if match:
            try:
                obj = json.loads(match.group(0))
                return obj.get("name"), obj.get("parameters", obj.get("arguments", {}))
            except Exception:
                return None
        return None
    try:
        obj = json.loads(match.group(1))
        return obj.get("name"), obj.get("parameters", obj.get("arguments", {}))
    except Exception:
        return None


def run_episode(ticket: dict, env: TriageEnvForTraining, max_turns: int = 20) -> dict:
    """Run one eval episode with the trained model."""
    tid = ticket.get("ticket_id", ticket.get("id", ""))
    obs_text = env.reset(ticket_id=tid)

    messages = [
        {"role": "system", "content": "You are an enterprise IT triage agent. Use tools to investigate, then submit_resolution."},
        {"role": "user",   "content": obs_text},
    ]

    for turn in range(max_turns):
        reply = generate_reply(messages)
        messages.append({"role": "assistant", "content": reply})

        parsed = parse_tool_call(reply)
        if parsed is None:
            # No tool call — model gave up
            break

        name, args = parsed
        tool_fn = getattr(env, name, None)
        if tool_fn is None:
            tool_result = f'{{"error": "Unknown tool: {name}"}}'
        else:
            try:
                tool_result = tool_fn(**args)
            except Exception as ex:
                tool_result = f'{{"error": "{ex}"}}'

        messages.append({"role": "tool", "name": name, "content": tool_result})

        if name == "submit_resolution" or env._state.submitted:
            break

    from server.rewards import reward_breakdown
    breakdown = reward_breakdown(env._state)
    return {
        "ticket_id": tid,
        "difficulty": ticket.get("difficulty", "medium"),
        "turns": turn + 1,
        "submitted": env._state.submitted,
        "rewards": breakdown,
    }


# Run eval
eval_results = []
n_eval = min(5, len(eval_tickets)) if SMOKE_TEST else len(eval_tickets)
print(f"Running eval on {n_eval} tickets{' (SMOKE_TEST)' if SMOKE_TEST else ''}...")

for i, ticket in enumerate(eval_tickets[:n_eval]):
    env = TriageEnvForTraining(corpus, ticket_index)
    result = run_episode(ticket, env)
    eval_results.append(result)
    print(f"  [{i+1}/{n_eval}] {result['ticket_id']} "
          f"total={result['rewards']['total']:.3f} "
          f"primary={result['rewards']['primary']:.1f}")

print("\nEval complete.")

In [ ]:
# Load baseline numbers
baseline_path = ASSETS_DIR / "baseline_eval.json"
with open(baseline_path) as f:
    baseline_data = json.load(f)

baseline_means = baseline_data["meta"]["mean_rewards"]

# Compute trained means
sub_rewards = ["primary", "grounding", "efficiency", "calibration", "format", "total"]
trained_means = {}
for key in sub_rewards:
    vals = [r["rewards"].get(key, 0.0) for r in eval_results]
    trained_means[key] = float(np.mean(vals)) if vals else 0.0

# Print comparison table
print(f"{'Sub-reward':<14} {'Baseline':>10} {'Trained':>10} {'Delta':>10}")
print("-" * 48)
for key in sub_rewards:
    b = baseline_means.get(key, 0.0)
    t = trained_means.get(key, 0.0)
    delta = t - b
    arrow = "▲" if delta > 0.01 else ("▼" if delta < -0.01 else "≈")
    print(f"{key:<14} {b:>10.4f} {t:>10.4f} {arrow} {delta:+.4f}")

# Comparison bar chart
plot_keys = ["primary", "grounding", "efficiency", "calibration", "total"]
x = np.arange(len(plot_keys))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars_b = ax.bar(x - width/2, [baseline_means.get(k, 0) for k in plot_keys],
                width, label="Baseline (pre-training)", color="#5b9bd5", alpha=0.85)
bars_t = ax.bar(x + width/2, [trained_means.get(k, 0) for k in plot_keys],
                width, label="Trained (GRPO)", color="#ed7d31", alpha=0.85)

ax.set_xlabel("Sub-reward", fontsize=12)
ax.set_ylabel("Mean reward", fontsize=12)
ax.set_title("Baseline vs GRPO-Trained: TriageAgent Eval Rewards", fontsize=13, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels([k.title() for k in plot_keys], fontsize=11)
ax.legend(fontsize=11)
ax.set_ylim(0, max(max(baseline_means.values()), max(trained_means.values())) * 1.25)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f"))
ax.grid(axis="y", alpha=0.35)

# Value labels
for bar in bars_b:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f"{bar.get_height():.2f}", ha="center", va="bottom", fontsize=8)
for bar in bars_t:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f"{bar.get_height():.2f}", ha="center", va="bottom", fontsize=8)

plt.tight_layout()
comparison_plot_path = PLOTS_DIR / "baseline_vs_trained.png"
plt.savefig(comparison_plot_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {comparison_plot_path}")

## 10. Next steps

- **If training is flat (steps 0–30):** increase `max_completion_length` to 8192 — most common cause.
- **If OOM:** reduce `num_generations=2`, set `vllm_gpu_memory_utilization=0.25`.
- **To push adapter to HF Hub:**
  ```python
  from huggingface_hub import HfApi
  api = HfApi()
  api.upload_folder(folder_path=adapter_path, repo_id="<yourname>/triage-agent-qwen3b")
  ```
- **HF Space:** the trained agent can be tested at `https://huggingface.co/spaces/yahid/triage_agent_env`
- **Plots** are saved to `assets/plots/` and embedded in the README.